In [ ]:
import pandas as pd
df=pd.read_csv('master_final.csv')

def star_to_sentiment(score):
  if score>=4:
    return 'positive'
  elif score==3:
    return 'neutral'
  else:
    return 'negative'
df['sentiment_stars']=df['score'].apply(star_to_sentiment)
df['sentiment_ai']=df['sentiment'].str.lower()

def sentiment_mismatch(row):
  stars=row['sentiment_stars']
  ai=row['sentiment_ai']
  if ai=='sem comentários' or pd.isna(ai):
    return 'sem comentários'

  if ai==stars:
    return 'aligned'
  elif stars=='positive' and ai=='negative':
    return 'hidden dissatisfied'
  elif stars=='negative' and ai=='positive':
    return 'mild complainer'
  else:
    return 'mild mismatch'
df['mismatch_type']=df.apply(sentiment_mismatch,axis=1)

mismatch_summary=df.groupby(['product_category_name','mismatch_type']).agg(customer_count=('order_id','count'),
                                                                           revenue=('price','sum'),
                                                                           avg_star_rating=('score','mean'),
                                                                           avg_ai_sentiment=('sentiment_score','mean')).reset_index()

mismatch_overall=df.groupby('mismatch_type').agg(customer_count=('order_id','count'),
                                                revenue=('price','sum')
                                                 ).reset_index()
mismatch_summary.to_csv('mismatch_summary.csv', index=False)
mismatch_overall.to_csv('mismatch_overall.csv', index=False)